# TensorData Quickstart

This notebook introduces the main ideas behind `TensorData` and shows a few practical examples based on the unit tests.

## What is TensorData?

`TensorData` is a unified data container used to pass data between nodes in a tensor-based pipeline.

In practice, it helps standardize how data is represented inside the system:

- features and targets are stored as tensors;
- different input formats can be converted into the same internal representation;
- preprocessing logic can be applied during creation, so downstream nodes receive consistent tensor data.

This makes it easier for nodes to communicate through a common data flow format instead of handling raw NumPy arrays, pandas tables, CSV files, text columns, categorical columns, and tensors separately.


## Imports

Adjust the import paths below to match your project structure.


In [6]:
import sys
sys.path.append('/workspaces/FEDOT')

In [7]:
import os
import numpy as np
import pandas as pd
import torch

# Example project imports
# Update these imports if your package layout is different.
from fedot.core.data.tensordata import TensorData
from fedot.core.data.ucr_loader import TSLoader
from fedot.core.utils import fedot_project_root


/workspaces/FEDOT/venv/lib/python3.10/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/workspaces/FEDOT/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Create TensorData from a NumPy array

A common case is creating `TensorData` directly from a numeric NumPy matrix.

In this example, the last column is treated as the target automatically, so:

- `features` becomes a tensor with all columns except the target;
- `target` becomes a tensor with one value per row.


In [8]:
features = np.random.rand(100, 10)

td = TensorData.create(
    features,
    backend_name="cpu"
)

print(type(td))
print(type(td.features))
print(td.features.shape)
print(td.target.shape)


<class 'fedot.core.data.tensordata.TensorData'>
<class 'torch.Tensor'>
torch.Size([100, 9])
torch.Size([100, 1])


## 2. Create TensorData from a CSV file

`TensorData.create(...)` can also load data from a file path.  
Here the target column is selected explicitly by name.


In [9]:
csv_path = f"{fedot_project_root()}/examples/real_cases/data/scoring/scoring_train.csv"

td_csv = TensorData.create(
    csv_path,
    backend_name="cpu",
    target_idx="target"
)

print(type(td_csv.features))
print(type(td_csv.target))
print(td_csv.features.shape)
print(td_csv.target.shape)


<class 'torch.Tensor'>
<class 'torch.Tensor'>
torch.Size([20000, 10])
torch.Size([20000, 1])


## 3. Create TensorData from a torch tensor

When the input is already a torch tensor, `TensorData` preserves it as tensor data.

This is useful when data has already been prepared by another tensor-based component.

In this example, no target is provided, so `target` remains `None`.


In [10]:
x_torch = torch.rand(100, 10)

td_torch = TensorData.create(
    x_torch,
    backend_name="cpu"
)

print(type(td_torch.features))
print(td_torch.features.shape)
print(td_torch.target)


<class 'torch.Tensor'>
torch.Size([100, 10])
None


## 4. Load an external time series dataset

The test suite also shows a case where a dataset is downloaded with `TSLoader`, then wrapped into `TensorData`.

This is a good example of how `TensorData` acts as a bridge between raw loaded data and the tensor pipeline.


In [11]:
name = "AbnormalHeartbeat"
X_train, y_train, X_test, y_test = TSLoader().download_by_url(dataset_name=name)

train_tensor = TensorData.create(
    X_train,
    target=y_train,
    backend_name="cpu"
)

test_tensor = TensorData.create(
    X_test,
    target=y_test,
    backend_name="cpu"
)

print(train_tensor.features.shape, train_tensor.target.shape)
print(test_tensor.features.shape, test_tensor.target.shape)


2026-03-20 17:05:07,606 - Reading data from /workspaces/FEDOT/temp_cache/temp_data/AbnormalHeartbeat
torch.Size([204, 18530]) torch.Size([204, 1])
torch.Size([205, 18530]) torch.Size([205, 1])


## 5. Text data: CSV to TensorData with embeddings

`TensorData` can preprocess text columns and convert them into embedding vectors during creation.

In this example:

- one column is treated as text;
- a sentence-transformer model generates dense embeddings;
- the result is stored as a 2D feature tensor.

This is convenient because downstream nodes receive numeric tensor features without needing to know how text embedding was produced.


In [12]:
spam_csv_path = f"{fedot_project_root()}/examples/real_cases/data/spam/spamham.csv"

df = pd.read_csv(spam_csv_path)

df.head()

,text,label
0,date wed NUMBER aug NUMBER NUMBER NUMBER NUMB...,0
1,martin a posted tassos papadopoulos the greek ...,0
2,man threatens explosion in moscow thursday aug...,0
3,klez the virus that won t die already the most...,0
4,in adding cream to spaghetti carbonara which ...,0


In [15]:
spam_csv_path = f"{fedot_project_root()}/examples/real_cases/data/spam/spamham.csv"

td_text = TensorData.create(
    spam_csv_path,
    backend_name="gpu",
    text_idx=0,
    embedding_strategy={
        "method": "sentence_transformer",
        "model_name": "all-distilroberta-v1",
        "batch_size": 3,
        "device": "cuda"
    },
    max_rows=10
)

print(td_text.features.shape)
print(td_text.target.shape)
print(td_text.features.device)


2026-03-20 17:07:47,226 - Using pandas instead of cudf. Failed to get values from cudf DataFrame.
2026-03-20 17:07:47,229 - Turning to cpu backend to get TensorData due to failed to convert features to cupy array
2026-03-20 17:07:47,232 - Load pretrained SentenceTransformer: all-distilroberta-v1
torch.Size([10, 768])
torch.Size([10, 1])
cuda:0


## 6. Mixed preprocessing: text + categorical + numeric features

One of the most important `TensorData` features is that preprocessing can be combined.

Below:

- `text1` and `text2` are embedded into dense vectors;
- `class` and `subclass` are label-encoded;
- `number` is kept as a numeric feature;
- all transformed parts are concatenated into one final feature tensor.

This allows heterogeneous raw input to become a single tensor representation.


In [16]:
X = np.array([
    ["date wed NUMBER aug NUMBER NUMBER NUMBER NUMBER NUMBER from chris garrigues cwg",
     "in adding cream to spaghetti carbonara which has the same effect on pasta", 1, "A", "DOP", 0],
    ["martin a posted tassos papadopoulos the greek sculptor behind",
     "i just had to jump in here as carbonara is one of my favourites to make", 1, "A", "DROP", 0],
    ["man threatens explosion in moscow thursday august NUMBER NUMBER NUMBER NUMBER pm",
     "in adding cream to spaghetti carbonara which has the same effect on pasta", 3, "B", "DOP", 1],
], dtype=object)

columns = ["text1", "text2", "number", "class", "subclass", "target"]

encoding_strategy = {
    "label": ["class", "subclass"]
}

embedding_strategy = {
    "method": "sentence_transformer",
    "model_name": "all-distilroberta-v1",
    "batch_size": 3,
    "device": "cpu",
}

td_mixed = TensorData.create(
    X,
    backend_name="cpu",
    features_names=columns,
    encoding_strategy=encoding_strategy,
    text_idx=["text1", "text2"],
    embedding_strategy=embedding_strategy
)

print(td_mixed.features.shape)


2026-03-20 17:08:11,170 - Load pretrained SentenceTransformer: all-distilroberta-v1
torch.Size([3, 1539])


## 7. Reuse fitted encoding in prediction mode

A useful pattern is to fit preprocessing on training data and reuse the same encoding configuration for inference.

This keeps feature transformations consistent between train and predict stages.


In [17]:
X_train = np.array([
    [100, "A", "C", 10],
    [500, "B", "D", 20],
    [100, "A", "D", 30],
], dtype=object)

X_test = np.array([
    [100, "A", "C"],
    [500, "B", "D"],
    [100, "A", "D"],
], dtype=object)

td_train = TensorData.create(
    X_train,
    backend_name="cpu",
    target_idx=3
)

td_test = TensorData.create(
    X_test,
    backend_name="cpu",
    state="predict",
    encoding_strategy=td_train.encoding_strategy,
)

print(td_train.features.shape)
print(td_test.features.shape)
print(torch.equal(td_test.features, td_train.features))


torch.Size([3, 3])
torch.Size([3, 3])
True


## 8. Lazy TensorData creation

Sometimes it is useful to postpone materialization.  
`create_lazy(...)` creates a lazy wrapper instead of constructing the tensor data immediately.

This can help avoid unnecessary work until the data is actually needed.


In [18]:
X_lazy = np.random.rand(10, 3)

lazy_td = TensorData.create_lazy(
    X_lazy,
    backend_name="cpu"
)

print(type(lazy_td))
print(lazy_td._data)

td_realized = lazy_td.get()

print(type(td_realized))
print(td_realized.features.shape)


<class 'fedot.core.data.tensordata.LazyTensor'>
None
<class 'fedot.core.data.tensordata.TensorData'>
torch.Size([10, 2])


## 9. Handling special cases: datetime columns and missing targets

The tests also cover important data-cleaning behaviors.

### Datetime features
Datetime columns can be included in the input table and processed during `TensorData` creation.

### Missing target values
Rows with missing target values can be dropped so that features and targets stay aligned.


In [19]:
df = pd.DataFrame({
    "date": pd.date_range("2022-01-01", periods=10, freq="D"),
    "feature1": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "target": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
})

td_datetime = TensorData.create(
    df,
    backend_name="cpu",
    target_idx="target"
)

print(td_datetime.features.shape)
print(td_datetime.target.shape)

X = np.array([
    [1.0, 2.0],
    [3.0, 4.0],
    [5.0, 6.0],
])

y = np.array([1.0, None, 0.0], dtype=object)

td_nan = TensorData.create(
    X,
    target=y,
    backend_name="cpu"
)

print(td_nan.features.shape)
print(td_nan.target.shape)


torch.Size([10, 2])
torch.Size([10, 1])
torch.Size([2, 2])
torch.Size([2, 1])


## Summary

`TensorData` provides a single tensor-oriented interface for different kinds of raw input.

The main practical benefits are:

- one common format for node-to-node data exchange;
- automatic conversion of raw sources into tensors;
- support for text, categorical, numeric, datetime, and file-based inputs;
- reusable preprocessing logic between training and prediction;
- optional lazy materialization.

This makes `TensorData` a convenient transport and preprocessing layer inside a pipeline built around tensor operations.
